# The following code cells illustrate how to use DynaTab for binary and multiclass classification, as well as regression, with or without Optuna-based hyperparameter tuning.

> **Citation.** If you use DynaTab in your work, please cite:  
> *Al Zadid Sultan Bin Habib, Gianfranco Doretto, and Donald A. Adjeroh.*  
> **DynaTab: Dynamic Feature Ordering as Neural Rewiring for High-Dimensional Tabular Data.**  
> In **AAAI 2026 First International Workshop on Neuro for AI & AI for Neuro: Towards Multi-Modal Natural Intelligence (NeuroAI) Workshop Proceedings (PMLR)**.
> Shorter citation: A. Z. S. B. Habib, G. Doretto, and D. A. Adjeroh, *“DynaTab: Dynamic Feature Ordering as Neural Rewiring for High-Dimensional Tabular Data,”* AAAI 2026 NeuroAI Workshop Proceedings (PMLR).

In [4]:
from dynatab import (
    DynaTabBinary, DynaTabMulti, DynaTabRegression,
    TrainConfig, ModelConfig, LossConfig,
    DFOConfig, run_dfo, reorder_and_evaluate,
    train_one_split, evaluate_split, cross_validate,
    CustomFeatureLoss, evaluate_predictions,
    OrderAwarePositionalEmbedding, DynamicMaskedAttention, create_dma_mask,
)

In [2]:
from dynatab import DynaTabBinary, DynaTabMulti, DynaTabRegression
from dynatab import TrainConfig, ModelConfig, LossConfig
from dynatab import train_one_split, evaluate_split, cross_validate

In [3]:
from dynatab.model import DynaTabBinary, DynaTabMulti, DynaTabRegression
from dynatab.trainer import train_one_split, evaluate_split, cross_validate
from dynatab.dfo import DFOConfig, run_dfo, reorder_and_evaluate
from dynatab.customloss import CustomFeatureLoss

In [4]:
def check():
    from dynatab import DynaTabBinary, DFOConfig, TrainConfig, CustomFeatureLoss
    from dynatab import run_dfo, train_one_split
    print("key imports OK")

check()

key imports OK


# The following code cells illustrate how to use DynaTab for binary classification or multiclass classification or regression, w\ or w\o Optuna tuning 

# Binary Classification || BUPA Liver Dataset || LDLSS

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

from dynatab import (
    DynaTabClassifier,
    DFOConfig,
    TrainConfig,
    LossConfig,
)

# === 1) Load & split off a held-out test set (10%) ===
df = pd.read_csv("bupa_liver_processed.csv")
X = df.drop(columns="label")
y = df["label"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.1,
    stratify=y,
    random_state=42,
)

# === 2) 5-fold CV on TRAIN_VAL, evaluate each fold on held-out TEST ===
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
val_accs, test_accs = [], []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_val, y_train_val), start=1):
    X_tr  = X_train_val.iloc[tr_idx].copy()
    y_tr  = y_train_val.iloc[tr_idx].copy()
    X_val = X_train_val.iloc[val_idx].copy()
    y_val = y_train_val.iloc[val_idx].copy()

    # --- configs ---
    dfo_cfg = DFOConfig(
        metric="manhattan",
        num_clusters=2,
        order="descending",
        mutation_prob=0.0,
        tolerance=0.001,
        seed=42,
    )

    loss_cfg = LossConfig(
        loss_mode="DFO",       # "standard" | "dispersion" | "DFO"
        lambda_disp=0.0,
        lambda_global=0.0,
    )

    train_cfg = TrainConfig(
        epochs=100,
        lr=1e-3,
        batch_size=256,
        print_every=20,
    )

    # === 3) Estimator (DFO + OPE + PIGL + DMA + backbone end-to-end) ===
    clf = DynaTabClassifier(
        task="binary",
        backbone="Transformer",
        embedding_dim=128,
        backbone_kwargs={
            # optional transformer params if your backbone supports them
            # "d_model": 256,
            # "nhead": 4,
            # "num_layers": 3,
            # "window_size": 64,
        },
        dfo_cfg=dfo_cfg,
        train_cfg=train_cfg,
        loss_cfg=loss_cfg,
        eval_metrics=["acc"],
        device=None,          # auto cuda/cpu
        standardize=True,     # estimator does train-only impute + standardize
    )

    # Fit using train split (optionally pass val to see training-time val prints/metrics)
    clf.fit(X_tr, y_tr, X_val=X_val, y_val=y_val)

    # Validation metrics for this fold
    val_metrics = clf.score(X_val, y_val, metrics=["acc"])
    val_acc = float(val_metrics.get("acc", np.nan))

    # Test metrics for this fold (same held-out test every fold)
    test_metrics = clf.score(X_test, y_test, metrics=["acc"])
    test_acc = float(test_metrics.get("acc", np.nan))

    print(f"Fold {fold} — Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}")

    val_accs.append(val_acc)
    test_accs.append(test_acc)

print()
print(f"5-Fold CV Validation Accuracy: {np.mean(val_accs):.4f} ± {np.std(val_accs):.4f}")
print(f"5-Fold CV     Test Accuracy: {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")

Epoch 001 | loss=0.658140
  val: {'acc': 0.6613}
Epoch 020 | loss=0.060873
  val: {'acc': 1.0}
Epoch 040 | loss=0.025655
  val: {'acc': 1.0}
Epoch 060 | loss=0.145371
  val: {'acc': 0.9839}
Epoch 080 | loss=0.014228
  val: {'acc': 1.0}
Epoch 100 | loss=0.009962
  val: {'acc': 1.0}
Fold 1 — Val Acc: 1.0000, Test Acc: 1.0000
Epoch 001 | loss=0.651886
  val: {'acc': 0.8226}
Epoch 020 | loss=0.057259
  val: {'acc': 1.0}
Epoch 040 | loss=0.026295
  val: {'acc': 1.0}
Epoch 060 | loss=0.016354
  val: {'acc': 1.0}
Epoch 080 | loss=0.010452
  val: {'acc': 1.0}
Epoch 100 | loss=0.007795
  val: {'acc': 1.0}
Fold 2 — Val Acc: 1.0000, Test Acc: 1.0000
Epoch 001 | loss=0.694190
  val: {'acc': 0.7419}
Epoch 020 | loss=0.072414
  val: {'acc': 1.0}
Epoch 040 | loss=0.027078
  val: {'acc': 1.0}
Epoch 060 | loss=0.015594
  val: {'acc': 1.0}
Epoch 080 | loss=0.009995
  val: {'acc': 1.0}
Epoch 100 | loss=0.007610
  val: {'acc': 1.0}
Fold 3 — Val Acc: 1.0000, Test Acc: 1.0000
Epoch 001 | loss=0.669168
  val

In [1]:
import dynatab, os
print(dynatab.__file__)
print(os.listdir(os.path.dirname(dynatab.__file__)))

C:\Users\ah00069\dynatab\__init__.py
['.ipynb_checkpoints', 'customloss.py', 'dfo.py', 'dma.py', 'estimator.py', 'metrics.py', 'model.py', 'ope.py', 'pigl.py', 'preprocess.py', 'seqprobinary.py', 'seqpromulti.py', 'seqproregression.py', 'trainer.py', '__init__.py', '__pycache__']


# Binary classification || Water Potability Dataset || MixedRegime

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

from dynatab import (
    DynaTabClassifier,
    DFOConfig,
    TrainConfig,
    LossConfig,
)

# ============================================================
# 1) Load
# ============================================================
df = pd.read_csv("water_potability_cleaned.csv")

X = df.drop(columns="Potability")
y = df["Potability"].astype(int)  # ensure 0/1 ints

# ============================================================
# 2) Split test (10%)
# ============================================================
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.1,
    stratify=y,
    random_state=42,
)

# ============================================================
# 3) 5-fold CV (estimator handles train-only impute + standardize)
# ============================================================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
val_accs, test_accs = [], []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_val, y_train_val), start=1):
    X_tr  = X_train_val.iloc[tr_idx].copy()
    y_tr  = y_train_val.iloc[tr_idx].copy()
    X_val = X_train_val.iloc[val_idx].copy()
    y_val = y_train_val.iloc[val_idx].copy()

    # ---- configs ----
    dfo_cfg = DFOConfig(
        metric="manhattan",
        num_clusters=2,
        order="ascending",
        mutation_prob=0.0,
        tolerance=0.001,
        seed=42,
    )

    loss_cfg = LossConfig(
        loss_mode="DFO",      # "standard" | "dispersion" | "DFO"
        lambda_disp=0.0,
        lambda_global=0.0,
    )

    train_cfg = TrainConfig(
        epochs=200,
        lr=1e-3,
        batch_size=256,
        print_every=20,
    )

    clf = DynaTabClassifier(
        task="binary",
        backbone="Transformer",
        embedding_dim=128,
        backbone_kwargs={
            # optional transformer params if your backbone supports them
            # "d_model": 256,
            # "nhead": 4,
            # "num_layers": 3,
            # "window_size": 64,
        },
        dfo_cfg=dfo_cfg,
        train_cfg=train_cfg,
        loss_cfg=loss_cfg,
        eval_metrics=["acc"],
        device=None,          # auto cuda/cpu
        standardize=True,     # train-only impute + standardize inside estimator
    )

    # Fit (DFO runs on X_tr only; val is reordered using train-derived cols)
    clf.fit(X_tr, y_tr, X_val=X_val, y_val=y_val)

    # Fold validation accuracy
    val_metrics = clf.score(X_val, y_val, metrics=["acc"])
    val_acc = float(val_metrics.get("acc", np.nan))

    # Held-out test accuracy (same test set each fold)
    test_metrics = clf.score(X_test, y_test, metrics=["acc"])
    test_acc = float(test_metrics.get("acc", np.nan))

    print(f"Fold {fold} — Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}")
    val_accs.append(val_acc)
    test_accs.append(test_acc)

print()
print(f"5-Fold CV Validation Accuracy: {np.mean(val_accs):.4f} ± {np.std(val_accs):.4f}")
print(f"5-Fold CV     Test Accuracy: {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")

Epoch 001 | loss=0.713861
  val: {'acc': 0.3898}
Epoch 020 | loss=0.631749
  val: {'acc': 0.6407}
Epoch 040 | loss=0.608688
  val: {'acc': 0.5881}
Epoch 060 | loss=0.573956
  val: {'acc': 0.6475}
Epoch 080 | loss=0.539651
  val: {'acc': 0.6203}
Epoch 100 | loss=0.541926
  val: {'acc': 0.6203}
Epoch 120 | loss=0.527519
  val: {'acc': 0.5898}
Epoch 140 | loss=0.506814
  val: {'acc': 0.6136}
Epoch 160 | loss=0.487538
  val: {'acc': 0.622}
Epoch 180 | loss=0.516332
  val: {'acc': 0.5441}
Epoch 200 | loss=0.454089
  val: {'acc': 0.6305}
Fold 1 — Val Acc: 0.6305, Test Acc: 0.5915
Epoch 001 | loss=0.707801
  val: {'acc': 0.5051}
Epoch 020 | loss=0.633177
  val: {'acc': 0.5593}
Epoch 040 | loss=0.593897
  val: {'acc': 0.6102}
Epoch 060 | loss=0.583189
  val: {'acc': 0.622}
Epoch 080 | loss=0.558074
  val: {'acc': 0.6627}
Epoch 100 | loss=0.556204
  val: {'acc': 0.6492}
Epoch 120 | loss=0.538789
  val: {'acc': 0.6237}
Epoch 140 | loss=0.541776
  val: {'acc': 0.6458}
Epoch 160 | loss=0.511206
  

# This is a demo run with Optuna tuning with 10 trials only. Please update the trial number for in depth hyperparameter search.

# Binary Classification || AI-D (case 5) dataset || w/ demo Optuna tuning || MixedRegime

In [1]:
import os, json
import numpy as np
import pandas as pd
import torch
import optuna

from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional
from sklearn.model_selection import StratifiedKFold

from dynatab import (
    DynaTabClassifier,
    DFOConfig,
    TrainConfig,
    LossConfig,
)

# -----------------------------
# Utils
# -----------------------------
def set_global_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_binary_labels(y: pd.Series) -> pd.Series:
    if y.dropna().isin([0, 1]).all():
        return y.astype(int)

    mapping = {
        "yes": 1, "no": 0,
        "true": 1, "false": 0,
        "positive": 1, "negative": 0,
        "pos": 1, "neg": 0,
        "alive": 1, "dead": 0,
    }
    y2 = y.astype(str).str.strip().str.lower()
    if y2.isin(mapping.keys()).all():
        return y2.map(mapping).astype(int)

    uniq = sorted(pd.unique(y.dropna()))
    if uniq == [1, 2]:
        return (y.astype(int) - 1).astype(int)

    raise ValueError(f"Status column not recognized as binary 0/1. Unique values: {sorted(pd.unique(y))}")


# -----------------------------
# Config
# -----------------------------
@dataclass
class OptunaHDLSSConfig:
    csv_path: str = "AI-d_case5.csv"
    label_col: str = "Status"

    n_trials: int = 150
    n_splits: int = 5
    seed: int = 42

    # training fixed
    epochs: int = 100
    lr: float = 1e-3
    weight_decay: float = 0.0
    batch_size: int = 32
    print_every: int = 0

    # metric to optimize
    metric: str = "acc"

    # device
    device: Optional[str] = None  # "cuda" | "cpu" | None(auto)

    # loss
    loss_mode: str = "standard"
    lambda_disp: float = 0.0
    lambda_global: float = 0.0

    # optuna storage/output
    out_dir: str = "./optuna_ai_d_case5"
    study_name: str = "ai_d_case5_transformer_dfo"


# -----------------------------
# Main experiment (Estimator-style objective)
# -----------------------------
class OptunaTransformerDFO:
    def __init__(self, cfg: OptunaHDLSSConfig):
        self.cfg = cfg
        os.makedirs(cfg.out_dir, exist_ok=True)

        if cfg.device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(cfg.device)

        set_global_seed(cfg.seed)

        df = pd.read_csv(cfg.csv_path)
        if cfg.label_col not in df.columns:
            raise ValueError(f"Label col '{cfg.label_col}' not found. Columns: {list(df.columns)[:10]}...")

        y = ensure_binary_labels(df[cfg.label_col])
        X = df.drop(columns=[cfg.label_col])

        # lightweight sanitation (estimator also sanitizes + imputes)
        X = X.replace([np.inf, -np.inf], np.nan)

        self.X = X
        self.y = y.astype(int)

        print("Device:", self.device)
        print("Loaded:", self.X.shape, "label counts:", self.y.value_counts().to_dict())

        self.kf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)

    # UPDATED to match your dynatab.dfo MetricName exactly
    def _suggest_dfo_params(self, trial: optuna.Trial) -> Dict[str, Any]:
        dfo_metric = trial.suggest_categorical(
            "dfo_metric",
            [
                "variance",
                "euclidean",
                "correlation",
                "manhattan",
                "kl_divergence",
                "mutual_info",
                "js_divergence",
                "wasserstein_distance",
            ],
        )
        dfo_num_clusters = trial.suggest_int("dfo_num_clusters", 2, 8)
        dfo_order = trial.suggest_categorical("dfo_order", ["ascending", "descending"])
        dfo_mut_prob = trial.suggest_float("dfo_mut_prob", 0.0, 0.7)
        dfo_tol = trial.suggest_float("dfo_tol", 1e-4, 5e-2, log=True)

        return dict(
            metric=dfo_metric,
            num_clusters=int(dfo_num_clusters),
            order=dfo_order,
            mutation_prob=float(dfo_mut_prob),
            tolerance=float(dfo_tol),
        )

    def _suggest_transformer_params(self, trial: optuna.Trial) -> Tuple[int, Dict[str, Any]]:
        embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128])

        d_model = trial.suggest_categorical("d_model", [64, 128, 256])
        nhead = trial.suggest_categorical("nhead", [2, 4, 8])
        num_layers = trial.suggest_int("num_layers", 1, 6)
        dropout = trial.suggest_float("dropout", 0.0, 0.4)

        window_size = trial.suggest_categorical("window_size", [-1, 64, 128, 256])
        window_size = None if window_size == -1 else int(window_size)

        backbone_kwargs = dict(
            d_model=int(d_model),
            nhead=int(nhead),
            num_layers=int(num_layers),
            dropout=float(dropout),
        )
        if window_size is not None:
            backbone_kwargs["window_size"] = window_size

        return int(embedding_dim), backbone_kwargs

    def objective(self, trial: optuna.Trial) -> float:
        cfg = self.cfg

        dfo_params = self._suggest_dfo_params(trial)
        embedding_dim, backbone_kwargs = self._suggest_transformer_params(trial)

        dfo_cfg = DFOConfig(
            metric=dfo_params["metric"],
            num_clusters=int(dfo_params["num_clusters"]),
            order=dfo_params["order"],
            mutation_prob=float(dfo_params["mutation_prob"]),
            tolerance=float(dfo_params["tolerance"]),
            seed=cfg.seed,
        )

        loss_cfg = LossConfig(
            loss_mode=cfg.loss_mode,
            lambda_disp=cfg.lambda_disp,
            lambda_global=cfg.lambda_global,
        )

        train_cfg = TrainConfig(
            epochs=cfg.epochs,
            lr=cfg.lr,
            batch_size=cfg.batch_size,
            weight_decay=cfg.weight_decay,
            print_every=cfg.print_every,
        )

        fold_scores = []

        for fold_id, (tr_idx, va_idx) in enumerate(self.kf.split(self.X, self.y), start=1):
            X_tr  = self.X.iloc[tr_idx].copy()
            y_tr  = self.y.iloc[tr_idx].copy()
            X_val = self.X.iloc[va_idx].copy()
            y_val = self.y.iloc[va_idx].copy()

            clf = DynaTabClassifier(
                task="binary",
                backbone="Transformer",
                embedding_dim=embedding_dim,
                backbone_kwargs=backbone_kwargs,
                dfo_cfg=dfo_cfg,
                train_cfg=train_cfg,
                loss_cfg=loss_cfg,
                eval_metrics=[cfg.metric],
                device=self.device,      # keep consistent with your original script
                standardize=True,        # fold-wise train-only preprocessing
            )

            clf.fit(X_tr, y_tr, X_val=X_val, y_val=y_val)

            val_metrics = clf.score(X_val, y_val, metrics=[cfg.metric])
            score = float(val_metrics.get(cfg.metric, np.nan))
            fold_scores.append(score)

            interim = float(np.nanmean(fold_scores))
            trial.report(interim, step=fold_id)
            if trial.should_prune():
                raise optuna.TrialPruned(f"Pruned at fold {fold_id} with interim {interim:.4f}")

        return float(np.nanmean(fold_scores))

    def run(self):
        sampler = optuna.samplers.TPESampler(seed=self.cfg.seed)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=2)

        study = optuna.create_study(
            direction="maximize",
            sampler=sampler,
            pruner=pruner,
            study_name=self.cfg.study_name,
        )

        study.optimize(self.objective, n_trials=self.cfg.n_trials, show_progress_bar=True)

        best = {
            "best_value": study.best_value,
            "best_params": study.best_params,
            "n_trials": len(study.trials),
        }

        out_path = os.path.join(self.cfg.out_dir, f"{self.cfg.study_name}_best.json")
        with open(out_path, "w") as f:
            json.dump(best, f, indent=2)

        print("\n=== DONE ===")
        print("Best value:", best["best_value"])
        print("Best params:", best["best_params"])
        print("Saved:", out_path)

        return study, best


# -----------------------------
# Run
# -----------------------------
cfg = OptunaHDLSSConfig(
    csv_path="AI-d_case5.csv",
    label_col="Status",
    n_trials=10,
    n_splits=5,
    epochs=100,
    batch_size=32,
    lr=1e-3,
    metric="acc",
    print_every=0,
    out_dir="./optuna_ai_d_case5",
    study_name="ai_d_case5_transformer_dfo_150t",
)

runner = OptunaTransformerDFO(cfg)
study, best = runner.run()

[I 2026-02-20 13:03:42,071] A new study created in memory with name: ai_d_case5_transformer_dfo_150t


Device: cuda
Loaded: (239, 393) label counts: {0: 167, 1: 72}


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-02-20 13:04:53,510] Trial 0 finished with value: 0.6985815763473511 and parameters: {'dfo_metric': 'euclidean', 'dfo_num_clusters': 6, 'dfo_order': 'ascending', 'dfo_mut_prob': 0.678936896513396, 'dfo_tol': 0.017649715848175724, 'embedding_dim': 32, 'd_model': 128, 'nhead': 4, 'num_layers': 2, 'dropout': 0.1465447373174767, 'window_size': 64}. Best is trial 0 with value: 0.6985815763473511.
[I 2026-02-20 13:24:26,845] Trial 1 finished with value: 0.40540780425071715 and parameters: {'dfo_metric': 'js_divergence', 'dfo_num_clusters': 4, 'dfo_order': 'descending', 'dfo_mut_prob': 0.3081067456177209, 'dfo_tol': 0.0002134899990195199, 'embedding_dim': 128, 'd_model': 128, 'nhead': 4, 'num_layers': 6, 'dropout': 0.31005312934444584, 'window_size': -1}. Best is trial 0 with value: 0.6985815763473511.
[I 2026-02-20 13:37:57,594] Trial 2 finished with value: 0.7529255390167237 and parameters: {'dfo_metric': 'js_divergence', 'dfo_num_clusters': 3, 'dfo_order': 'ascending', 'dfo_mut_prob

C:\Users\ah00069\dynatab\dfo.py:167: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  metric_matrix = torch.var(X_tensor.unsqueeze(2) - X_tensor.unsqueeze(1), dim=0).abs()


[I 2026-02-20 13:52:20,661] Trial 4 finished with value: 0.6986702084541321 and parameters: {'dfo_metric': 'variance', 'dfo_num_clusters': 7, 'dfo_order': 'ascending', 'dfo_mut_prob': 0.07703634716937373, 'dfo_tol': 0.0004122780057519721, 'embedding_dim': 128, 'd_model': 128, 'nhead': 8, 'num_layers': 6, 'dropout': 0.1292811728083021, 'window_size': 256}. Best is trial 2 with value: 0.7529255390167237.
[I 2026-02-20 13:54:07,155] Trial 5 finished with value: 0.7239361763000488 and parameters: {'dfo_metric': 'variance', 'dfo_num_clusters': 2, 'dfo_order': 'descending', 'dfo_mut_prob': 0.16769332346688068, 'dfo_tol': 0.0002460746712417508, 'embedding_dim': 64, 'd_model': 128, 'nhead': 2, 'num_layers': 4, 'dropout': 0.2143098736299034, 'window_size': 64}. Best is trial 2 with value: 0.7529255390167237.
[I 2026-02-20 13:55:05,874] Trial 6 finished with value: 0.732358169555664 and parameters: {'dfo_metric': 'correlation', 'dfo_num_clusters': 6, 'dfo_order': 'descending', 'dfo_mut_prob': 0.

# Multiclass || ADNI (AD123) dataset || w/ demo Optuna tuning || MixedRegime

In [2]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import optuna

from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional
from sklearn.model_selection import StratifiedKFold

from dynatab import (
    DynaTabClassifier,
    DFOConfig,
    TrainConfig,
    LossConfig,
)

# -----------------------------
# Utils
# -----------------------------
def set_global_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_multiclass_labels(y: pd.Series) -> pd.Series:
    """
    Make labels 0..C-1 ints.
    Accepts:
      - already 0..C-1
      - 1..C  (common "1,2,3" AD123)
      - strings (will factorize)
    """
    if pd.api.types.is_numeric_dtype(y):
        yn = y.astype(int)
        uniq = sorted(pd.unique(yn.dropna()))
        # already 0..C-1
        if uniq and uniq[0] == 0:
            return yn
        # 1..C -> 0..C-1
        if uniq and uniq[0] == 1:
            return (yn - 1).astype(int)
        return yn.astype(int)

    # string labels -> factorize to 0..C-1
    codes, uniques = pd.factorize(y.astype(str).str.strip())
    # factorize returns -1 for NaN; disallow here
    if (codes < 0).any():
        raise ValueError("Found missing labels after factorize. Please clean label column.")
    return pd.Series(codes, index=y.index).astype(int)


# -----------------------------
# Config
# -----------------------------
@dataclass
class OptunaADNIConfig:
    csv_path: str = "ADNI_AD123.csv"
    label_col: str = "label"          # CHANGE if your label column name differs

    n_trials: int = 100
    n_splits: int = 5
    seed: int = 42

    # multiclass
    num_classes: int = 3              # AD123 -> 3 classes (adjust if needed)

    # training fixed
    epochs: int = 100
    lr: float = 1e-3
    weight_decay: float = 0.0
    batch_size: int = 64
    print_every: int = 0              # silent trainer; we print fold-level progress

    # metric to optimize
    metric: str = "acc"               # or "macro_f1" if your evaluator supports it

    # device
    device: Optional[str] = None

    # loss
    loss_mode: str = "standard"       # start with standard for Optuna; can switch later
    lambda_disp: float = 0.0
    lambda_global: float = 0.0

    # output
    out_dir: str = "./optuna_adni_ad123"
    study_name: str = "adni_ad123_multiclass_dynatab"


# -----------------------------
# Main experiment
# -----------------------------
class OptunaADNIMulticlass:
    def __init__(self, cfg: OptunaADNIConfig):
        self.cfg = cfg
        os.makedirs(cfg.out_dir, exist_ok=True)

        if cfg.device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(cfg.device)

        set_global_seed(cfg.seed)

        df = pd.read_csv(cfg.csv_path)
        if cfg.label_col not in df.columns:
            raise ValueError(
                f"Label col '{cfg.label_col}' not found. "
                f"Available columns (first 20): {list(df.columns)[:20]}"
            )

        y = ensure_multiclass_labels(df[cfg.label_col])
        X = df.drop(columns=[cfg.label_col])

        # light sanitation (estimator also imputes/standardizes if enabled)
        X = X.replace([np.inf, -np.inf], np.nan)

        self.X = X
        self.y = y.astype(int)

        uniq = sorted(pd.unique(self.y))
        print("Device:", self.device)
        print("Loaded:", self.X.shape, "classes:", uniq, "counts:", self.y.value_counts().to_dict())

        # sanity: ensure labels within [0, C-1]
        if self.y.min() < 0 or self.y.max() >= cfg.num_classes:
            raise ValueError(
                f"Labels out of range for num_classes={cfg.num_classes}. "
                f"Got min={self.y.min()}, max={self.y.max()}, uniq={uniq}"
            )

        self.kf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)

    def _suggest_dfo_params(self, trial: optuna.Trial) -> Dict[str, Any]:
        # fast metrics (avoid the CPU-heavy distributional metrics in Optuna)
        dfo_metric = trial.suggest_categorical(
            "dfo_metric",
            ["variance", "euclidean", "correlation", "manhattan"],
        )
        dfo_num_clusters = trial.suggest_int("dfo_num_clusters", 2, 6)
        dfo_order = trial.suggest_categorical("dfo_order", ["ascending", "descending"])
        dfo_mut_prob = trial.suggest_float("dfo_mut_prob", 0.0, 0.7)
        dfo_tol = trial.suggest_float("dfo_tol", 1e-4, 5e-2, log=True)

        return dict(
            metric=dfo_metric,
            num_clusters=int(dfo_num_clusters),
            order=dfo_order,
            mutation_prob=float(dfo_mut_prob),
            tolerance=float(dfo_tol),
        )

    def _suggest_transformer_params(self, trial: optuna.Trial) -> Tuple[int, Dict[str, Any]]:
        embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128])

        d_model = trial.suggest_categorical("d_model", [64, 128, 256])
        nhead = trial.suggest_categorical("nhead", [2, 4, 8])
        num_layers = trial.suggest_int("num_layers", 1, 6)
        dropout = trial.suggest_float("dropout", 0.0, 0.4)

        window_size = trial.suggest_categorical("window_size", [-1, 64, 128, 256])
        window_size = None if window_size == -1 else int(window_size)

        backbone_kwargs = dict(
            d_model=int(d_model),
            nhead=int(nhead),
            num_layers=int(num_layers),
            dropout=float(dropout),
        )
        if window_size is not None:
            backbone_kwargs["window_size"] = window_size

        return int(embedding_dim), backbone_kwargs

    def objective(self, trial: optuna.Trial) -> float:
        cfg = self.cfg

        dfo_params = self._suggest_dfo_params(trial)
        embedding_dim, backbone_kwargs = self._suggest_transformer_params(trial)

        dfo_cfg = DFOConfig(
            metric=dfo_params["metric"],
            num_clusters=dfo_params["num_clusters"],
            order=dfo_params["order"],
            mutation_prob=dfo_params["mutation_prob"],
            tolerance=dfo_params["tolerance"],
            seed=cfg.seed,
        )

        loss_cfg = LossConfig(
            loss_mode=cfg.loss_mode,
            lambda_disp=cfg.lambda_disp,
            lambda_global=cfg.lambda_global,
        )

        train_cfg = TrainConfig(
            epochs=cfg.epochs,
            lr=cfg.lr,
            batch_size=cfg.batch_size,
            weight_decay=cfg.weight_decay,
            print_every=cfg.print_every,
        )

        fold_scores = []

        print(
            f"\n[Trial {trial.number}] "
            f"DFO(metric={dfo_cfg.metric}, K={dfo_cfg.num_clusters}, order={dfo_cfg.order}, "
            f"mut={dfo_cfg.mutation_prob:.2f}, tol={dfo_cfg.tolerance:.1e}) | "
            f"Transformer(emb={embedding_dim}, d_model={backbone_kwargs['d_model']}, "
            f"heads={backbone_kwargs['nhead']}, layers={backbone_kwargs['num_layers']}, "
            f"drop={backbone_kwargs['dropout']:.2f}, win={backbone_kwargs.get('window_size', None)})"
        )

        for fold_id, (tr_idx, va_idx) in enumerate(self.kf.split(self.X, self.y), start=1):
            t0 = time.time()

            X_tr  = self.X.iloc[tr_idx].copy()
            y_tr  = self.y.iloc[tr_idx].copy()
            X_val = self.X.iloc[va_idx].copy()
            y_val = self.y.iloc[va_idx].copy()

            clf = DynaTabClassifier(
                task="multiclass",
                num_classes=cfg.num_classes,
                backbone="Transformer",
                embedding_dim=embedding_dim,
                backbone_kwargs=backbone_kwargs,
                dfo_cfg=dfo_cfg,
                train_cfg=train_cfg,
                loss_cfg=loss_cfg,
                eval_metrics=[cfg.metric],
                device=self.device,
                standardize=True,
            )

            clf.fit(X_tr, y_tr, X_val=X_val, y_val=y_val)

            val_metrics = clf.score(X_val, y_val, metrics=[cfg.metric])
            score = float(val_metrics.get(cfg.metric, np.nan))
            fold_scores.append(score)

            interim = float(np.nanmean(fold_scores))
            dt = time.time() - t0
            print(f"  Fold {fold_id}/{cfg.n_splits} | {cfg.metric}={score:.4f} | interim={interim:.4f} | {dt:.1f}s")

            trial.report(interim, step=fold_id)
            if trial.should_prune():
                raise optuna.TrialPruned(f"Pruned at fold {fold_id} with interim {interim:.4f}")

        return float(np.nanmean(fold_scores))

    def run(self):
        sampler = optuna.samplers.TPESampler(seed=self.cfg.seed)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)

        study = optuna.create_study(
            direction="maximize",
            sampler=sampler,
            pruner=pruner,
            study_name=self.cfg.study_name,
        )

        study.optimize(self.objective, n_trials=self.cfg.n_trials, show_progress_bar=True)

        best = {
            "best_value": float(study.best_value),
            "best_params": study.best_params,
            "n_trials": len(study.trials),
        }

        out_path = os.path.join(self.cfg.out_dir, f"{self.cfg.study_name}_best.json")
        with open(out_path, "w") as f:
            json.dump(best, f, indent=2)

        print("\n=== DONE ===")
        print("Best value:", best["best_value"])
        print("Best params:", best["best_params"])
        print("Saved:", out_path)

        return study, best


# -----------------------------
# Run
# -----------------------------
cfg = OptunaADNIConfig(
    csv_path="ADNI_AD123.csv",
    label_col="AD123",   # target label
    n_trials=10,
    n_splits=5,
    epochs=100,
    batch_size=64,
    lr=1e-3,
    metric="acc",
    out_dir="./optuna_adni_ad123",
    study_name="adni_ad123_multiclass_dynatab",
    num_classes=3,
)

runner = OptunaADNIMulticlass(cfg)
study, best = runner.run()

[I 2026-02-20 14:58:32,196] A new study created in memory with name: adni_ad123_multiclass_dynatab


Device: cuda
Loaded: (177, 263) classes: [0, 1, 2] counts: {2: 108, 1: 36, 0: 33}


  0%|          | 0/10 [00:00<?, ?it/s]


[Trial 0] DFO(metric=euclidean, K=2, order=ascending, mut=0.61, tol=4.2e-03) | Transformer(emb=128, d_model=64, heads=8, layers=3, drop=0.12, win=None)
  Fold 1/5 | acc=0.9444 | interim=0.9444 | 11.0s
  Fold 2/5 | acc=1.0000 | interim=0.9722 | 10.5s
  Fold 3/5 | acc=0.8000 | interim=0.9148 | 10.5s
  Fold 4/5 | acc=0.6571 | interim=0.8504 | 10.5s
  Fold 5/5 | acc=0.7714 | interim=0.8346 | 10.7s
[I 2026-02-20 14:59:25,443] Trial 0 finished with value: 0.8346031904220581 and parameters: {'dfo_metric': 'euclidean', 'dfo_num_clusters': 2, 'dfo_order': 'ascending', 'dfo_mut_prob': 0.6063233020424545, 'dfo_tol': 0.004191711516695206, 'embedding_dim': 128, 'd_model': 64, 'nhead': 8, 'num_layers': 3, 'dropout': 0.11649165607921677, 'window_size': -1}. Best is trial 0 with value: 0.8346031904220581.

[Trial 1] DFO(metric=euclidean, K=4, order=descending, mut=0.12, tol=1.5e-04) | Transformer(emb=64, d_model=256, heads=8, layers=1, drop=0.36, win=64)
  Fold 1/5 | acc=0.9722 | interim=0.9722 | 6.6

C:\Users\ah00069\dynatab\dfo.py:173: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  X_norm = (X_tensor - X_tensor.mean(dim=0)) / (X_tensor.std(dim=0) + 1e-10)


  Fold 4/5 | acc=1.0000 | interim=0.9224 | 6.9s


C:\Users\ah00069\dynatab\dfo.py:173: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  X_norm = (X_tensor - X_tensor.mean(dim=0)) / (X_tensor.std(dim=0) + 1e-10)


  Fold 5/5 | acc=0.9429 | interim=0.9265 | 6.9s
[I 2026-02-20 15:00:33,235] Trial 2 finished with value: 0.9265079498291016 and parameters: {'dfo_metric': 'correlation', 'dfo_num_clusters': 6, 'dfo_order': 'ascending', 'dfo_mut_prob': 0.6453119645161818, 'dfo_tol': 0.00017331598058558715, 'embedding_dim': 128, 'd_model': 256, 'nhead': 8, 'num_layers': 1, 'dropout': 0.32087879230161587, 'window_size': 64}. Best is trial 2 with value: 0.9265079498291016.

[Trial 3] DFO(metric=euclidean, K=5, order=descending, mut=0.08, tol=2.1e-02) | Transformer(emb=32, d_model=256, heads=4, layers=1, drop=0.29, win=128)
  Fold 1/5 | acc=1.0000 | interim=1.0000 | 5.6s
  Fold 2/5 | acc=0.9722 | interim=0.9861 | 5.6s
  Fold 3/5 | acc=0.7143 | interim=0.8955 | 5.8s
  Fold 4/5 | acc=0.9714 | interim=0.9145 | 5.7s
  Fold 5/5 | acc=1.0000 | interim=0.9316 | 5.7s
[I 2026-02-20 15:01:01,619] Trial 3 finished with value: 0.931587302684784 and parameters: {'dfo_metric': 'euclidean', 'dfo_num_clusters': 5, 'dfo_ord

# Regression || Cargo dataset || w/ demo Optuna tuning || MixedRegime

In [5]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import optuna

from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional
from sklearn.model_selection import KFold

from dynatab import (
    DynaTabRegressor,
    DFOConfig,
    TrainConfig,
    LossConfig,
)

# -----------------------------
# Utils
# -----------------------------
def set_global_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_regression_target(y: pd.Series) -> pd.Series:
    y = pd.to_numeric(y, errors="coerce")
    if y.isna().any():
        raise ValueError(f"Target has NaNs after numeric coercion: {y.isna().sum()} rows. Clean label column.")
    return y.astype(float)

# -----------------------------
# Config
# -----------------------------
@dataclass
class OptunaCargoRegConfig:
    csv_path: str = "cargo_cleaned_labelencoded.csv"
    label_col: str = "o_dlv_e"

    n_trials: int = 50
    n_splits: int = 5
    seed: int = 42

    # training fixed
    epochs: int = 100
    lr: float = 1e-3
    weight_decay: float = 0.0
    batch_size: int = 64
    print_every: int = 0

    # metric to optimize
    metric: str = "r2"

    # device
    device: Optional[str] = None

    # loss
    # recommend "standard" during optuna for stability
    loss_mode: str = "standard"
    lambda_disp: float = 0.0
    lambda_global: float = 0.0

    # output
    out_dir: str = "./optuna_cargo_reg"
    study_name: str = "cargo_reg_dynatab_r2"


# -----------------------------
# Main experiment
# -----------------------------
class OptunaCargoRegression:
    def __init__(self, cfg: OptunaCargoRegConfig):
        self.cfg = cfg
        os.makedirs(cfg.out_dir, exist_ok=True)

        if cfg.device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(cfg.device)

        set_global_seed(cfg.seed)

        df = pd.read_csv(cfg.csv_path)
        if cfg.label_col not in df.columns:
            raise ValueError(
                f"Label col '{cfg.label_col}' not found. "
                f"Available columns (first 20): {list(df.columns)[:20]}"
            )

        y = ensure_regression_target(df[cfg.label_col])
        X = df.drop(columns=[cfg.label_col])

        # light sanitation (estimator also imputes/standardizes if enabled)
        X = X.replace([np.inf, -np.inf], np.nan)

        self.X = X
        self.y = y

        print("Device:", self.device)
        print("Loaded:", self.X.shape, "target stats:", dict(
            min=float(self.y.min()),
            max=float(self.y.max()),
            mean=float(self.y.mean()),
            std=float(self.y.std(ddof=0)),
        ))

        self.kf = KFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)

    def _suggest_dfo_params(self, trial: optuna.Trial) -> Dict[str, Any]:
        # keep optuna fast: avoid CPU-heavy distributional metrics for DFO like KL/JS Divergence or Wassertian 
        dfo_metric = trial.suggest_categorical(
            "dfo_metric",
            ["variance", "euclidean", "correlation", "manhattan"],
        )
        dfo_num_clusters = trial.suggest_int("dfo_num_clusters", 2, 6)
        dfo_order = trial.suggest_categorical("dfo_order", ["ascending", "descending"])
        dfo_mut_prob = trial.suggest_float("dfo_mut_prob", 0.0, 0.7)
        dfo_tol = trial.suggest_float("dfo_tol", 1e-4, 5e-2, log=True)

        return dict(
            metric=dfo_metric,
            num_clusters=int(dfo_num_clusters),
            order=dfo_order,
            mutation_prob=float(dfo_mut_prob),
            tolerance=float(dfo_tol),
        )

    def _suggest_transformer_params(self, trial: optuna.Trial) -> Tuple[int, Dict[str, Any]]:
        embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128])

        d_model = trial.suggest_categorical("d_model", [64, 128, 256])
        nhead = trial.suggest_categorical("nhead", [2, 4, 8])
        num_layers = trial.suggest_int("num_layers", 1, 6)
        dropout = trial.suggest_float("dropout", 0.0, 0.4)

        window_size = trial.suggest_categorical("window_size", [-1, 64, 128, 256])
        window_size = None if window_size == -1 else int(window_size)

        backbone_kwargs = dict(
            d_model=int(d_model),
            nhead=int(nhead),
            num_layers=int(num_layers),
            dropout=float(dropout),
        )
        if window_size is not None:
            backbone_kwargs["window_size"] = window_size

        return int(embedding_dim), backbone_kwargs

    def objective(self, trial: optuna.Trial) -> float:
        cfg = self.cfg

        dfo_params = self._suggest_dfo_params(trial)
        embedding_dim, backbone_kwargs = self._suggest_transformer_params(trial)

        dfo_cfg = DFOConfig(
            metric=dfo_params["metric"],
            num_clusters=dfo_params["num_clusters"],
            order=dfo_params["order"],
            mutation_prob=dfo_params["mutation_prob"],
            tolerance=dfo_params["tolerance"],
            seed=cfg.seed,
        )

        loss_cfg = LossConfig(
            loss_mode=cfg.loss_mode,
            lambda_disp=cfg.lambda_disp,
            lambda_global=cfg.lambda_global,
        )

        train_cfg = TrainConfig(
            epochs=cfg.epochs,
            lr=cfg.lr,
            batch_size=cfg.batch_size,
            weight_decay=cfg.weight_decay,
            print_every=cfg.print_every,
        )

        fold_scores = []

        print(
            f"\n[Trial {trial.number}] "
            f"DFO(metric={dfo_cfg.metric}, K={dfo_cfg.num_clusters}, order={dfo_cfg.order}, "
            f"mut={dfo_cfg.mutation_prob:.2f}, tol={dfo_cfg.tolerance:.1e}) | "
            f"Transformer(emb={embedding_dim}, d_model={backbone_kwargs['d_model']}, "
            f"heads={backbone_kwargs['nhead']}, layers={backbone_kwargs['num_layers']}, "
            f"drop={backbone_kwargs['dropout']:.2f}, win={backbone_kwargs.get('window_size', None)})"
        )

        for fold_id, (tr_idx, va_idx) in enumerate(self.kf.split(self.X), start=1):
            t0 = time.time()

            X_tr  = self.X.iloc[tr_idx].copy()
            y_tr  = self.y.iloc[tr_idx].copy()
            X_val = self.X.iloc[va_idx].copy()
            y_val = self.y.iloc[va_idx].copy()

            reg = DynaTabRegressor(
                backbone="Transformer",
                embedding_dim=embedding_dim,
                backbone_kwargs=backbone_kwargs,
                dfo_cfg=dfo_cfg,
                train_cfg=train_cfg,
                loss_cfg=loss_cfg,
                eval_metrics=[cfg.metric],
                device=self.device,
                standardize=True,
            )

            reg.fit(X_tr, y_tr, X_val=X_val, y_val=y_val)

            val_metrics = reg.score(X_val, y_val, metrics=[cfg.metric])
            score = float(val_metrics.get(cfg.metric, np.nan))
            fold_scores.append(score)

            interim = float(np.nanmean(fold_scores))
            dt = time.time() - t0
            print(f"  Fold {fold_id}/{cfg.n_splits} | {cfg.metric}={score:.4f} | interim={interim:.4f} | {dt:.1f}s")

            trial.report(interim, step=fold_id)
            if trial.should_prune():
                raise optuna.TrialPruned(f"Pruned at fold {fold_id} with interim {interim:.4f}")

        return float(np.nanmean(fold_scores))

    def run(self):
        sampler = optuna.samplers.TPESampler(seed=self.cfg.seed)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)

        study = optuna.create_study(
            direction="maximize",
            sampler=sampler,
            pruner=pruner,
            study_name=self.cfg.study_name,
        )

        study.optimize(self.objective, n_trials=self.cfg.n_trials, show_progress_bar=True)

        best = {
            "best_value": float(study.best_value),
            "best_params": study.best_params,
            "n_trials": len(study.trials),
        }

        out_path = os.path.join(self.cfg.out_dir, f"{self.cfg.study_name}_best.json")
        with open(out_path, "w") as f:
            json.dump(best, f, indent=2)

        print("\n=== DONE ===")
        print("Best value:", best["best_value"])
        print("Best params:", best["best_params"])
        print("Saved:", out_path)

        return study, best


# -----------------------------
# Run
# -----------------------------
cfg = OptunaCargoRegConfig(
    csv_path="cargo_cleaned_labelencoded.csv",
    label_col="o_dlv_e",
    n_trials=10,
    n_splits=5,
    epochs=100,
    batch_size=64,
    lr=1e-3,
    metric="r2",
    out_dir="./optuna_cargo_reg",
    study_name="cargo_reg_dynatab_r2",
)

runner = OptunaCargoRegression(cfg)
study, best = runner.run()

[I 2026-02-20 15:29:25,388] A new study created in memory with name: cargo_reg_dynatab_r2


Device: cuda
Loaded: (3943, 97) target stats: {'min': 1.0, 'max': 560130.0, 'mean': 3697.930382957139, 'std': 14418.829877632463}


  0%|          | 0/10 [00:00<?, ?it/s]


[Trial 0] DFO(metric=euclidean, K=2, order=ascending, mut=0.61, tol=4.2e-03) | Transformer(emb=128, d_model=64, heads=8, layers=3, drop=0.12, win=None)
  Fold 1/5 | r2=-0.0470 | interim=-0.0470 | 51.8s
  Fold 2/5 | r2=-0.0557 | interim=-0.0514 | 52.0s
  Fold 3/5 | r2=-0.0125 | interim=-0.0384 | 51.8s
  Fold 4/5 | r2=-0.0919 | interim=-0.0518 | 51.9s
  Fold 5/5 | r2=-0.0113 | interim=-0.0437 | 51.8s
[I 2026-02-20 15:33:44,752] Trial 0 finished with value: -0.043672585487365724 and parameters: {'dfo_metric': 'euclidean', 'dfo_num_clusters': 2, 'dfo_order': 'ascending', 'dfo_mut_prob': 0.6063233020424545, 'dfo_tol': 0.004191711516695206, 'embedding_dim': 128, 'd_model': 64, 'nhead': 8, 'num_layers': 3, 'dropout': 0.11649165607921677, 'window_size': -1}. Best is trial 0 with value: -0.043672585487365724.

[Trial 1] DFO(metric=euclidean, K=4, order=descending, mut=0.12, tol=1.5e-04) | Transformer(emb=64, d_model=256, heads=8, layers=1, drop=0.36, win=64)
  Fold 1/5 | r2=-0.2754 | interim=-

# Binary Classification || Adult dataset || demo w/ Optuna tuning || LDHSS

In [6]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import optuna

from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional
from sklearn.model_selection import train_test_split

from dynatab import DynaTabClassifier, DFOConfig, TrainConfig, LossConfig


# -----------------------------
# Utils
# -----------------------------
def set_global_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_binary_labels(y: pd.Series) -> np.ndarray:
    # If already 0/1
    if y.dropna().isin([0, 1]).all():
        return y.astype(int).to_numpy()

    # If categorical strings -> codes
    if not pd.api.types.is_numeric_dtype(y):
        return y.astype("category").cat.codes.to_numpy()

    # Numeric but not 0/1 (e.g., {1,2})
    yn = y.astype(int)
    uniq = sorted(pd.unique(yn))
    if uniq == [1, 2]:
        return (yn - 1).astype(int).to_numpy()

    # fallback: category codes
    return y.astype("category").cat.codes.to_numpy()

def encode_object_columns(X: pd.DataFrame) -> pd.DataFrame:
    """
    If your adultcensus.csv has string columns, this makes it numeric.
    Uses one-hot encoding for object/category cols.
    """
    obj_cols = [c for c in X.columns if X[c].dtype == "object" or str(X[c].dtype).startswith("category")]
    if len(obj_cols) == 0:
        return X
    return pd.get_dummies(X, columns=obj_cols, drop_first=False)


# -----------------------------
# Config
# -----------------------------
@dataclass
class OptunaAdultConfig:
    file: str = "adultcensus.csv"
    target_col: str = "income"

    seed: int = 42
    n_trials: int = 30

    # training
    epochs: int = 60
    lr: float = 1e-3
    weight_decay: float = 0.0
    batch_size: int = 256
    print_every: int = 0

    # metric
    metric: str = "acc"  # change to "auc" if supported

    # loss
    loss_mode: str = "standard"
    lambda_disp: float = 0.0
    lambda_global: float = 0.0

    # output
    out_dir: str = "./optuna_adult"
    study_name: str = "adult_dynatab_optuna"


# -----------------------------
# Main
# -----------------------------
class OptunaAdultBinary:
    def __init__(self, cfg: OptunaAdultConfig):
        self.cfg = cfg
        os.makedirs(cfg.out_dir, exist_ok=True)

        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")

        set_global_seed(cfg.seed)

        # --- Load & Split 80/10/10 (your exact logic) ---
        df = pd.read_csv(cfg.file)
        if cfg.target_col not in df.columns:
            raise ValueError(f"target_col='{cfg.target_col}' not found. columns={list(df.columns)[:20]}")

        X = df.drop(columns=[cfg.target_col])
        y = ensure_binary_labels(df[cfg.target_col])

        # If needed, encode non-numeric columns (safe even if already numeric)
        X = encode_object_columns(X)

        # sanitize inf
        X = X.replace([np.inf, -np.inf], np.nan)

        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y, test_size=0.10, stratify=y, random_state=cfg.seed
        )
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_trainval, y_trainval,
            test_size=0.1111,  # 0.1111*0.9 ≈ 0.10 overall
            stratify=y_trainval,
            random_state=cfg.seed
        )

        # store
        self.X_train = X_train.reset_index(drop=True)
        self.y_train = pd.Series(y_train).reset_index(drop=True)

        self.X_valid = X_valid.reset_index(drop=True)
        self.y_valid = pd.Series(y_valid).reset_index(drop=True)

        self.X_test = X_test.reset_index(drop=True)
        self.y_test = pd.Series(y_test).reset_index(drop=True)

        print("Device:", self.device)
        print("Shapes:", {
            "train": self.X_train.shape,
            "valid": self.X_valid.shape,
            "test":  self.X_test.shape
        })
        print("Label counts:", pd.Series(y).value_counts().to_dict())

    def _suggest_dfo_params(self, trial: optuna.Trial) -> Dict[str, Any]:
        # fast DFO metrics for Optuna
        metric = trial.suggest_categorical("dfo_metric", ["variance", "euclidean", "correlation", "manhattan"])
        num_clusters = trial.suggest_int("dfo_num_clusters", 2, 6)
        order = trial.suggest_categorical("dfo_order", ["ascending", "descending"])
        mut_prob = trial.suggest_float("dfo_mut_prob", 0.0, 0.6)
        tol = trial.suggest_float("dfo_tol", 1e-4, 5e-2, log=True)
        return dict(metric=metric, num_clusters=num_clusters, order=order, mutation_prob=mut_prob, tolerance=tol)

    def _suggest_transformer_params(self, trial: optuna.Trial) -> Tuple[int, Dict[str, Any]]:
        embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128])

        d_model = trial.suggest_categorical("d_model", [64, 128, 256])
        nhead = trial.suggest_categorical("nhead", [2, 4, 8])
        num_layers = trial.suggest_int("num_layers", 1, 6)
        dropout = trial.suggest_float("dropout", 0.0, 0.4)

        window_size = trial.suggest_categorical("window_size", [-1, 64, 128, 256])
        window_size = None if window_size == -1 else int(window_size)

        kw = dict(d_model=int(d_model), nhead=int(nhead), num_layers=int(num_layers), dropout=float(dropout))
        if window_size is not None:
            kw["window_size"] = window_size

        return int(embedding_dim), kw

    def objective(self, trial: optuna.Trial) -> float:
        cfg = self.cfg

        dfo_p = self._suggest_dfo_params(trial)
        embedding_dim, backbone_kwargs = self._suggest_transformer_params(trial)

        dfo_cfg = DFOConfig(
            metric=dfo_p["metric"],
            num_clusters=int(dfo_p["num_clusters"]),
            order=dfo_p["order"],
            mutation_prob=float(dfo_p["mutation_prob"]),
            tolerance=float(dfo_p["tolerance"]),
            seed=cfg.seed,
        )

        loss_cfg = LossConfig(
            loss_mode=cfg.loss_mode,
            lambda_disp=cfg.lambda_disp,
            lambda_global=cfg.lambda_global,
        )

        train_cfg = TrainConfig(
            epochs=cfg.epochs,
            lr=cfg.lr,
            batch_size=cfg.batch_size,
            weight_decay=cfg.weight_decay,
            print_every=cfg.print_every,
        )

        t0 = time.time()
        clf = DynaTabClassifier(
            task="binary",
            backbone="Transformer",
            embedding_dim=embedding_dim,
            backbone_kwargs=backbone_kwargs,
            dfo_cfg=dfo_cfg,
            train_cfg=train_cfg,
            loss_cfg=loss_cfg,
            eval_metrics=[cfg.metric],
            device=self.device,
            standardize=True,
        )

        # Fit on TRAIN, evaluate on VALID
        clf.fit(self.X_train, self.y_train, X_val=self.X_valid, y_val=self.y_valid)

        val_metrics = clf.score(self.X_valid, self.y_valid, metrics=[cfg.metric])
        val_score = float(val_metrics.get(cfg.metric, np.nan))

        dt = time.time() - t0
        print(
            f"[Trial {trial.number}] {cfg.metric}={val_score:.4f} | "
            f"DFO({dfo_cfg.metric},K={dfo_cfg.num_clusters},{dfo_cfg.order},mut={dfo_cfg.mutation_prob:.2f}) | "
            f"emb={embedding_dim}, d_model={backbone_kwargs['d_model']}, heads={backbone_kwargs['nhead']}, "
            f"layers={backbone_kwargs['num_layers']}, drop={backbone_kwargs['dropout']:.2f} | {dt:.1f}s"
        )

        return val_score

    def run(self):
        sampler = optuna.samplers.TPESampler(seed=self.cfg.seed)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=0)

        study = optuna.create_study(
            direction="maximize",
            sampler=sampler,
            pruner=pruner,
            study_name=self.cfg.study_name,
        )

        study.optimize(self.objective, n_trials=self.cfg.n_trials, show_progress_bar=True)

        best = {
            "best_value": float(study.best_value),
            "best_params": study.best_params,
            "n_trials": len(study.trials),
        }

        out_path = os.path.join(self.cfg.out_dir, f"{self.cfg.study_name}_best.json")
        with open(out_path, "w") as f:
            json.dump(best, f, indent=2)

        print("\n=== BEST (val) ===")
        print(best)

        # --- Refit on TRAIN+VALID using best params, evaluate TEST ---
        bp = best["best_params"]

        dfo_cfg = DFOConfig(
            metric=bp["dfo_metric"],
            num_clusters=int(bp["dfo_num_clusters"]),
            order=bp["dfo_order"],
            mutation_prob=float(bp["dfo_mut_prob"]),
            tolerance=float(bp["dfo_tol"]),
            seed=self.cfg.seed,
        )

        window_size = bp["window_size"]
        window_size = None if int(window_size) == -1 else int(window_size)

        backbone_kwargs = dict(
            d_model=int(bp["d_model"]),
            nhead=int(bp["nhead"]),
            num_layers=int(bp["num_layers"]),
            dropout=float(bp["dropout"]),
        )
        if window_size is not None:
            backbone_kwargs["window_size"] = window_size

        train_cfg = TrainConfig(
            epochs=self.cfg.epochs,
            lr=self.cfg.lr,
            batch_size=self.cfg.batch_size,
            weight_decay=self.cfg.weight_decay,
            print_every=0,
        )

        loss_cfg = LossConfig(
            loss_mode=self.cfg.loss_mode,
            lambda_disp=self.cfg.lambda_disp,
            lambda_global=self.cfg.lambda_global,
        )

        X_tr_full = pd.concat([self.X_train, self.X_valid], axis=0).reset_index(drop=True)
        y_tr_full = pd.concat([self.y_train, self.y_valid], axis=0).reset_index(drop=True)

        clf = DynaTabClassifier(
            task="binary",
            backbone="Transformer",
            embedding_dim=int(bp["embedding_dim"]),
            backbone_kwargs=backbone_kwargs,
            dfo_cfg=dfo_cfg,
            train_cfg=train_cfg,
            loss_cfg=loss_cfg,
            eval_metrics=[self.cfg.metric],
            device=self.device,
            standardize=True,
        )

        clf.fit(X_tr_full, y_tr_full)

        test_metrics = clf.score(self.X_test, self.y_test, metrics=[self.cfg.metric])
        print("\n=== TEST ===")
        print(test_metrics)

        return study, best, test_metrics


# -----------------------------
# Run
# -----------------------------
cfg = OptunaAdultConfig(
    file="adultcensus.csv",
    target_col="income",
    n_trials=10,
    epochs=60,
    batch_size=256,
    lr=1e-3,
    metric="acc",
    out_dir="./optuna_adult",
    study_name="adult_dynatab_optuna",
)

runner = OptunaAdultBinary(cfg)
study, best, test_metrics = runner.run()

[I 2026-02-20 16:33:58,304] A new study created in memory with name: adult_dynatab_optuna


Device: cuda
Shapes: {'train': (26048, 14), 'valid': (3256, 14), 'test': (3257, 14)}
Label counts: {0: 24720, 1: 7841}


  0%|          | 0/10 [00:00<?, ?it/s]

[Trial 0] acc=0.8028 | DFO(euclidean,K=2,ascending,mut=0.52) | emb=128, d_model=64, heads=8, layers=3, drop=0.12 | 53.6s
[I 2026-02-20 16:34:51,896] Trial 0 finished with value: 0.8028255105018616 and parameters: {'dfo_metric': 'euclidean', 'dfo_num_clusters': 2, 'dfo_order': 'ascending', 'dfo_mut_prob': 0.519705687464961, 'dfo_tol': 0.004191711516695206, 'embedding_dim': 128, 'd_model': 64, 'nhead': 8, 'num_layers': 3, 'dropout': 0.11649165607921677, 'window_size': -1}. Best is trial 0 with value: 0.8028255105018616.
[Trial 1] acc=0.7230 | DFO(euclidean,K=4,descending,mut=0.10) | emb=64, d_model=256, heads=8, layers=1, drop=0.36 | 38.4s
[I 2026-02-20 16:35:30,344] Trial 1 finished with value: 0.7229729890823364 and parameters: {'dfo_metric': 'euclidean', 'dfo_num_clusters': 4, 'dfo_order': 'descending', 'dfo_mut_prob': 0.10231447421237491, 'dfo_tol': 0.00014982086432155476, 'embedding_dim': 64, 'd_model': 256, 'nhead': 8, 'num_layers': 1, 'dropout': 0.3637281608315128, 'window_size': 

# Multiclass || CIFAR10 ResNet Features || w/ Demo Optuna tuning || HDHSS || Use DAE or LSTM for High Dimensionality (if you have computational constraints) || Transformer or Mamba will require more computational power but will provide better performance

In [2]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import optuna

from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional, List
from sklearn.model_selection import StratifiedKFold

from dynatab import DynaTabClassifier, DFOConfig, TrainConfig, LossConfig


# =========================
# Settings
# =========================
FILE = "cifar10_resnet50_features.csv"   # adjust if needed
TARGET_COL = "label"


# =========================
# Utils
# =========================
def set_global_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_int_labels(y: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(y):
        return y.astype(int)
    codes, _ = pd.factorize(y.astype(str).str.strip())
    if (codes < 0).any():
        raise ValueError("Found missing labels after factorize. Please clean label column.")
    return pd.Series(codes, index=y.index).astype(int)

def infer_num_classes(y: pd.Series) -> int:
    uniq = sorted(pd.unique(y))
    if len(uniq) < 2:
        raise ValueError(f"Need >=2 classes, got uniq={uniq}")
    return int(len(uniq))


# =========================
# Config
# =========================
@dataclass
class OptunaCifarFeatConfig:
    file: str = FILE
    target_col: str = TARGET_COL

    seed: int = 42
    n_trials: int = 30
    n_splits: int = 5

    # training
    epochs: int = 40
    lr: float = 1e-3
    weight_decay: float = 0.0
    batch_size: int = 256
    print_every: int = 0

    # metric
    metric: str = "acc"

    # loss
    loss_mode: str = "standard"
    lambda_disp: float = 0.0
    lambda_global: float = 0.0

    # output
    out_dir: str = "./optuna_cifar10_resnet50_features_dae"
    study_name: str = "cifar10_resnet50_features_dynatab_dae"


# =========================
# Optuna Runner
# =========================
class OptunaCifarFeaturesDAE:
    def __init__(self, cfg: OptunaCifarFeatConfig):
        self.cfg = cfg
        os.makedirs(cfg.out_dir, exist_ok=True)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        set_global_seed(cfg.seed)

        df = pd.read_csv(cfg.file)
        if cfg.target_col not in df.columns:
            raise ValueError(
                f"TARGET_COL='{cfg.target_col}' not found. "
                f"Available columns (first 20): {list(df.columns)[:20]}"
            )

        y = ensure_int_labels(df[cfg.target_col])
        X = df.drop(columns=[cfg.target_col])

        # sanitize
        X = X.replace([np.inf, -np.inf], np.nan)

        # force numeric if any non-numeric slipped in
        for c in X.columns:
            if not pd.api.types.is_numeric_dtype(X[c]):
                X[c] = pd.to_numeric(X[c], errors="coerce")

        self.X = X.reset_index(drop=True)
        self.y = y.reset_index(drop=True)
        self.num_classes = infer_num_classes(self.y)

        print("Device:", self.device)
        print("Loaded:", self.X.shape, "| num_classes:", self.num_classes)
        print("Label counts (top 15):", self.y.value_counts().head(15).to_dict())

        self.kf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)

    def _suggest_dfo_params(self, trial: optuna.Trial) -> Dict[str, Any]:
        # keep Optuna fast: avoid CPU-heavy distributional metrics
        metric = trial.suggest_categorical("dfo_metric", ["variance", "euclidean", "correlation", "manhattan"])
        num_clusters = trial.suggest_int("dfo_num_clusters", 2, 6)
        order = trial.suggest_categorical("dfo_order", ["ascending", "descending"])
        mut_prob = trial.suggest_float("dfo_mut_prob", 0.0, 0.6)
        tol = trial.suggest_float("dfo_tol", 1e-4, 5e-2, log=True)
        return dict(metric=metric, num_clusters=num_clusters, order=order, mutation_prob=mut_prob, tolerance=tol)

    def _suggest_dae_params(self, trial: optuna.Trial) -> Tuple[int, Dict[str, Any]]:
        """
        DAE backbone tuning.

        IMPORTANT:
        - These kwargs must be accepted by your SequentialProcessor* DAE implementation.
        - If your DAE uses different names, just rename keys here.
        """
        embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128])

        hidden_dim = trial.suggest_categorical("dae_hidden_dim", [64, 128, 256, 512])
        num_layers = trial.suggest_int("dae_num_layers", 1, 6)
        dropout = trial.suggest_float("dae_dropout", 0.0, 0.4)

        # common in denoising autoencoders (if supported)
        noise_std = trial.suggest_float("dae_noise_std", 0.0, 0.3)

        backbone_kwargs = dict(
            hidden_dim=int(hidden_dim),
            num_layers=int(num_layers),
            dropout=float(dropout),
            noise_std=float(noise_std),
        )

        return int(embedding_dim), backbone_kwargs

    def _make_estimator(
        self,
        *,
        embedding_dim: int,
        backbone_kwargs: Dict[str, Any],
        dfo_cfg: DFOConfig,
        train_cfg: TrainConfig,
        loss_cfg: LossConfig,
    ) -> DynaTabClassifier:
        return DynaTabClassifier(
            task="multiclass",
            num_classes=self.num_classes,
            backbone="DAE",  # <-- switch
            embedding_dim=embedding_dim,
            backbone_kwargs=backbone_kwargs,
            dfo_cfg=dfo_cfg,
            train_cfg=train_cfg,
            loss_cfg=loss_cfg,
            eval_metrics=[self.cfg.metric],
            device=self.device,
            standardize=True,
        )

    def _cv_eval(self, *, embedding_dim: int, backbone_kwargs: Dict[str, Any], dfo_cfg: DFOConfig,
                train_cfg: TrainConfig, loss_cfg: LossConfig) -> List[float]:
        scores: List[float] = []
        for fold_id, (tr_idx, va_idx) in enumerate(self.kf.split(self.X, self.y), start=1):
            X_tr = self.X.iloc[tr_idx].copy()
            y_tr = self.y.iloc[tr_idx].copy()
            X_va = self.X.iloc[va_idx].copy()
            y_va = self.y.iloc[va_idx].copy()

            clf = self._make_estimator(
                embedding_dim=embedding_dim,
                backbone_kwargs=backbone_kwargs,
                dfo_cfg=dfo_cfg,
                train_cfg=train_cfg,
                loss_cfg=loss_cfg,
            )
            clf.fit(X_tr, y_tr, X_val=X_va, y_val=y_va)
            m = clf.score(X_va, y_va, metrics=[self.cfg.metric])
            scores.append(float(m.get(self.cfg.metric, np.nan)))
        return scores

    def objective(self, trial: optuna.Trial) -> float:
        cfg = self.cfg

        dfo_p = self._suggest_dfo_params(trial)
        embedding_dim, backbone_kwargs = self._suggest_dae_params(trial)

        dfo_cfg = DFOConfig(
            metric=dfo_p["metric"],
            num_clusters=int(dfo_p["num_clusters"]),
            order=dfo_p["order"],
            mutation_prob=float(dfo_p["mutation_prob"]),
            tolerance=float(dfo_p["tolerance"]),
            seed=cfg.seed,
        )

        loss_cfg = LossConfig(
            loss_mode=cfg.loss_mode,
            lambda_disp=cfg.lambda_disp,
            lambda_global=cfg.lambda_global,
        )

        train_cfg = TrainConfig(
            epochs=cfg.epochs,
            lr=cfg.lr,
            batch_size=cfg.batch_size,
            weight_decay=cfg.weight_decay,
            print_every=cfg.print_every,
        )

        t0 = time.time()
        fold_scores = []
        for fold_id, (tr_idx, va_idx) in enumerate(self.kf.split(self.X, self.y), start=1):
            X_tr = self.X.iloc[tr_idx].copy()
            y_tr = self.y.iloc[tr_idx].copy()
            X_va = self.X.iloc[va_idx].copy()
            y_va = self.y.iloc[va_idx].copy()

            clf = self._make_estimator(
                embedding_dim=embedding_dim,
                backbone_kwargs=backbone_kwargs,
                dfo_cfg=dfo_cfg,
                train_cfg=train_cfg,
                loss_cfg=loss_cfg,
            )
            clf.fit(X_tr, y_tr, X_val=X_va, y_val=y_va)

            m = clf.score(X_va, y_va, metrics=[cfg.metric])
            s = float(m.get(cfg.metric, np.nan))
            fold_scores.append(s)

            interim = float(np.nanmean(fold_scores))
            trial.report(interim, step=fold_id)
            if trial.should_prune():
                raise optuna.TrialPruned(f"Pruned at fold {fold_id} with interim {interim:.4f}")

        mean_score = float(np.nanmean(fold_scores))
        dt = time.time() - t0
        print(f"[Trial {trial.number}] mean {cfg.metric}={mean_score:.4f} | DAE | {dt:.1f}s")
        return mean_score

    def run(self):
        sampler = optuna.samplers.TPESampler(seed=self.cfg.seed)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)

        study = optuna.create_study(
            direction="maximize",
            sampler=sampler,
            pruner=pruner,
            study_name=self.cfg.study_name,
        )

        study.optimize(self.objective, n_trials=self.cfg.n_trials, show_progress_bar=True)

        best = {
            "best_value": float(study.best_value),
            "best_params": study.best_params,
            "n_trials": len(study.trials),
        }

        out_path = os.path.join(self.cfg.out_dir, f"{self.cfg.study_name}_best.json")
        with open(out_path, "w") as f:
            json.dump(best, f, indent=2)

        print("\n=== BEST (Optuna objective mean over folds) ===")
        print(best)
        print("Saved:", out_path)

        # ---- Final evaluation: re-run CV using BEST params to get mean ± std ----
        bp = best["best_params"]

        dfo_cfg = DFOConfig(
            metric=bp["dfo_metric"],
            num_clusters=int(bp["dfo_num_clusters"]),
            order=bp["dfo_order"],
            mutation_prob=float(bp["dfo_mut_prob"]),
            tolerance=float(bp["dfo_tol"]),
            seed=self.cfg.seed,
        )

        backbone_kwargs = dict(
            hidden_dim=int(bp["dae_hidden_dim"]),
            num_layers=int(bp["dae_num_layers"]),
            dropout=float(bp["dae_dropout"]),
            noise_std=float(bp["dae_noise_std"]),
        )

        train_cfg = TrainConfig(
            epochs=self.cfg.epochs,
            lr=self.cfg.lr,
            batch_size=self.cfg.batch_size,
            weight_decay=self.cfg.weight_decay,
            print_every=0,
        )

        loss_cfg = LossConfig(
            loss_mode=self.cfg.loss_mode,
            lambda_disp=self.cfg.lambda_disp,
            lambda_global=self.cfg.lambda_global,
        )

        fold_scores = self._cv_eval(
            embedding_dim=int(bp["embedding_dim"]),
            backbone_kwargs=backbone_kwargs,
            dfo_cfg=dfo_cfg,
            train_cfg=train_cfg,
            loss_cfg=loss_cfg,
        )

        mean_ = float(np.nanmean(fold_scores))
        std_ = float(np.nanstd(fold_scores))
        print("\n=== FINAL (best params) ===")
        print(f"{self.cfg.metric}: {mean_:.4f} ± {std_:.4f}  (over {self.cfg.n_splits} folds)")
        print("Per-fold:", [round(s, 4) for s in fold_scores])

        return study, best, fold_scores


# =========================
# Run
# =========================
cfg = OptunaCifarFeatConfig(
    file=FILE,
    target_col=TARGET_COL,
    n_trials=2,
    n_splits=5,
    epochs=40,
    batch_size=256,
    lr=1e-3,
    metric="acc",
    out_dir="./optuna_cifar10_resnet50_features_dae",
    study_name="cifar10_resnet50_features_dynatab_dae",
)

runner = OptunaCifarFeaturesDAE(cfg)
study, best, fold_scores = runner.run()

[I 2026-02-20 17:31:58,233] A new study created in memory with name: cifar10_resnet50_features_dynatab_dae


Device: cuda
Loaded: (11000, 2048) | num_classes: 10
Label counts (top 15): {1: 1126, 5: 1115, 2: 1104, 7: 1104, 0: 1100, 3: 1095, 4: 1092, 9: 1090, 8: 1090, 6: 1084}


  0%|          | 0/2 [00:00<?, ?it/s]

[Trial 0] mean acc=0.2550 | DAE | 217.4s
[I 2026-02-20 17:35:35,651] Trial 0 finished with value: 0.2550000011920929 and parameters: {'dfo_metric': 'euclidean', 'dfo_num_clusters': 2, 'dfo_order': 'ascending', 'dfo_mut_prob': 0.519705687464961, 'dfo_tol': 0.004191711516695206, 'embedding_dim': 128, 'dae_hidden_dim': 64, 'dae_num_layers': 2, 'dae_dropout': 0.20990257265289514, 'dae_noise_std': 0.12958350559263473}. Best is trial 0 with value: 0.2550000011920929.
[Trial 1] mean acc=0.2565 | DAE | 217.8s
[I 2026-02-20 17:39:13,443] Trial 1 finished with value: 0.2564545452594757 and parameters: {'dfo_metric': 'euclidean', 'dfo_num_clusters': 3, 'dfo_order': 'descending', 'dfo_mut_prob': 0.11980426929501584, 'dfo_tol': 0.0024428866967349987, 'embedding_dim': 128, 'dae_hidden_dim': 512, 'dae_num_layers': 5, 'dae_dropout': 0.12184550766934828, 'dae_noise_std': 0.029301634201915158}. Best is trial 1 with value: 0.2564545452594757.

=== BEST (Optuna objective mean over folds) ===
{'best_value'

# Binary classification || Colon dataset || w/ Demo Optuna tuning || HDLSS || Use DAE or LSTM for High Dimensionality (if you have computational constraints) || Transformer or Mamba will require more computational power but will provide better performance

In [3]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import optuna

from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional, List
from sklearn.model_selection import StratifiedKFold

from dynatab import (
    DynaTabClassifier,
    DFOConfig,
    TrainConfig,
    LossConfig,
)

# -----------------------------
# Utils
# -----------------------------
def set_global_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_binary_01(y: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(y):
        yn = y.astype(int)
        uniq = sorted(pd.unique(yn.dropna()))
        if uniq == [0, 1]:
            return yn
        if uniq == [1, 2]:
            return (yn - 1).astype(int)

    codes, _ = pd.factorize(y.astype(str).str.strip())
    if (codes < 0).any():
        raise ValueError("Found missing labels after factorize.")
    uniq = sorted(pd.unique(codes))
    if len(uniq) != 2:
        raise ValueError(f"Expected binary labels, got {uniq}")
    return pd.Series(codes, index=y.index).astype(int)

def make_5x5_splits(X: pd.DataFrame, y: pd.Series, repeats: int, n_splits: int, seed: int):
    """Return a list of (train_idx, val_idx) for repeat-stratified kfold."""
    splits = []
    for r in range(repeats):
        rep_seed = seed + 1000 * r
        kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rep_seed)
        for tr_idx, va_idx in kf.split(X, y):
            splits.append((tr_idx, va_idx))
    return splits

# -----------------------------
# Config
# -----------------------------
@dataclass
class OptunaColonLSTMConfig:
    csv_path: str = "coloncancer_encoded.csv"
    label_col: str = "label"

    # 5x5 CV
    repeats: int = 5
    n_splits: int = 5
    seed: int = 42

    # optuna
    n_trials: int = 30
    study_name: str = "colon_lstm_5x5_dynatab"
    out_dir: str = "./optuna_colon_lstm"

    # training fixed-ish (can tune epochs too if you want)
    epochs: int = 120
    lr: float = 1e-3
    weight_decay: float = 0.0
    batch_size: int = 32
    print_every: int = 0

    # loss
    loss_mode: str = "standard"   # keep standard for tuning speed/stability
    lambda_disp: float = 0.0
    lambda_global: float = 0.0

    # metric
    metric: str = "acc"

    # device
    device: Optional[str] = None  # None => auto


# -----------------------------
# Main Optuna runner
# -----------------------------
class OptunaColonLSTM5x5:
    def __init__(self, cfg: OptunaColonLSTMConfig):
        self.cfg = cfg
        os.makedirs(cfg.out_dir, exist_ok=True)

        if cfg.device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(cfg.device)

        set_global_seed(cfg.seed)

        df = pd.read_csv(cfg.csv_path)
        if cfg.label_col not in df.columns:
            raise ValueError(f"Label col '{cfg.label_col}' not found. Columns (first 30): {list(df.columns)[:30]}")

        y = ensure_binary_01(df[cfg.label_col])
        X = df.drop(columns=[cfg.label_col])

        # light sanitation; estimator will also sanitize/impute if enabled
        X = X.replace([np.inf, -np.inf], np.nan)
        for c in X.columns:
            if not pd.api.types.is_numeric_dtype(X[c]):
                X[c] = pd.to_numeric(X[c], errors="coerce")

        self.X = X
        self.y = y.astype(int)

        print("Device:", self.device)
        print("Loaded:", self.X.shape, "| label counts:", self.y.value_counts().to_dict())

        # Precompute 5x5 splits ONCE so every trial is comparable
        self.splits = make_5x5_splits(self.X, self.y, cfg.repeats, cfg.n_splits, cfg.seed)
        print(f"Prepared {len(self.splits)} CV folds ({cfg.repeats}x{cfg.n_splits}).")

    # ---- search space ----
    def _suggest_dfo(self, trial: optuna.Trial) -> DFOConfig:
        metric = trial.suggest_categorical("dfo_metric", ["variance", "euclidean", "correlation", "manhattan"])
        num_clusters = trial.suggest_int("dfo_num_clusters", 2, 6)
        order = trial.suggest_categorical("dfo_order", ["ascending", "descending"])
        mutation_prob = trial.suggest_float("dfo_mut_prob", 0.0, 0.7)
        tolerance = trial.suggest_float("dfo_tol", 1e-4, 5e-2, log=True)

        return DFOConfig(
            metric=metric,
            num_clusters=int(num_clusters),
            order=order,
            mutation_prob=float(mutation_prob),
            tolerance=float(tolerance),
            seed=self.cfg.seed,
        )

    def _suggest_lstm(self, trial: optuna.Trial) -> Tuple[int, Dict[str, Any]]:
        # token embedding dim for OPE/PIGL
        embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128])

        # LSTM hyperparams (make sure these keys match your LSTM backbone implementation)
        hidden_size = trial.suggest_categorical("lstm_hidden_size", [64, 128, 256])
        num_layers = trial.suggest_int("lstm_num_layers", 1, 3)
        dropout = trial.suggest_float("lstm_dropout", 0.0, 0.4)

        backbone_kwargs = dict(
            hidden_size=int(hidden_size),
            num_layers=int(num_layers),
            dropout=float(dropout),
            # optional (enable only if your LSTM supports it):
            # "bidirectional": trial.suggest_categorical("lstm_bidirectional", [False, True]),
        )
        return int(embedding_dim), backbone_kwargs

    # ---- one evaluation pass over all 25 folds ----
    def _eval_params(self, dfo_cfg: DFOConfig, embedding_dim: int, backbone_kwargs: Dict[str, Any]) -> Tuple[float, float, List[float]]:
        cfg = self.cfg

        loss_cfg = LossConfig(
            loss_mode=cfg.loss_mode,
            lambda_disp=cfg.lambda_disp,
            lambda_global=cfg.lambda_global,
        )

        train_cfg = TrainConfig(
            epochs=cfg.epochs,
            lr=cfg.lr,
            batch_size=cfg.batch_size,
            weight_decay=cfg.weight_decay,
            print_every=cfg.print_every,
        )

        scores = []
        for i, (tr_idx, va_idx) in enumerate(self.splits, start=1):
            X_tr = self.X.iloc[tr_idx].copy()
            y_tr = self.y.iloc[tr_idx].copy()
            X_va = self.X.iloc[va_idx].copy()
            y_va = self.y.iloc[va_idx].copy()

            clf = DynaTabClassifier(
                task="binary",
                backbone="LSTM",
                embedding_dim=embedding_dim,
                backbone_kwargs=backbone_kwargs,
                dfo_cfg=dfo_cfg,
                train_cfg=train_cfg,
                loss_cfg=loss_cfg,
                eval_metrics=[cfg.metric],
                device=self.device,
                standardize=True,
            )

            clf.fit(X_tr, y_tr, X_val=X_va, y_val=y_va)
            m = clf.score(X_va, y_va, metrics=[cfg.metric])
            s = float(m.get(cfg.metric, np.nan))
            scores.append(s)

        mean_ = float(np.nanmean(scores))
        std_ = float(np.nanstd(scores))
        return mean_, std_, scores

    # ---- optuna objective ----
    def objective(self, trial: optuna.Trial) -> float:
        cfg = self.cfg

        dfo_cfg = self._suggest_dfo(trial)
        embedding_dim, backbone_kwargs = self._suggest_lstm(trial)

        # Optional speed guardrails for HDLSS:
        # - If too many clusters for tiny n, penalize early
        # (you can remove this if you want)
        if dfo_cfg.num_clusters > 6:
            raise optuna.TrialPruned("Too many clusters for HDLSS tuning range")

        print(
            f"\n[Trial {trial.number}] "
            f"DFO(metric={dfo_cfg.metric}, K={dfo_cfg.num_clusters}, order={dfo_cfg.order}, "
            f"mut={dfo_cfg.mutation_prob:.2f}, tol={dfo_cfg.tolerance:.1e}) | "
            f"LSTM(emb={embedding_dim}, hidden={backbone_kwargs['hidden_size']}, "
            f"layers={backbone_kwargs['num_layers']}, drop={backbone_kwargs['dropout']:.2f})"
        )

        # Evaluate all folds; report intermediate for pruning
        scores = []
        for i, (tr_idx, va_idx) in enumerate(self.splits, start=1):
            t0 = time.time()

            X_tr = self.X.iloc[tr_idx].copy()
            y_tr = self.y.iloc[tr_idx].copy()
            X_va = self.X.iloc[va_idx].copy()
            y_va = self.y.iloc[va_idx].copy()

            loss_cfg = LossConfig(
                loss_mode=cfg.loss_mode,
                lambda_disp=cfg.lambda_disp,
                lambda_global=cfg.lambda_global,
            )
            train_cfg = TrainConfig(
                epochs=cfg.epochs,
                lr=cfg.lr,
                batch_size=cfg.batch_size,
                weight_decay=cfg.weight_decay,
                print_every=cfg.print_every,
            )

            clf = DynaTabClassifier(
                task="binary",
                backbone="LSTM",
                embedding_dim=embedding_dim,
                backbone_kwargs=backbone_kwargs,
                dfo_cfg=dfo_cfg,
                train_cfg=train_cfg,
                loss_cfg=loss_cfg,
                eval_metrics=[cfg.metric],
                device=self.device,
                standardize=True,
            )

            clf.fit(X_tr, y_tr, X_val=X_va, y_val=y_va)
            m = clf.score(X_va, y_va, metrics=[cfg.metric])
            s = float(m.get(cfg.metric, np.nan))
            scores.append(s)

            interim = float(np.nanmean(scores))
            dt = time.time() - t0
            print(f"  Fold {i}/{len(self.splits)} | {cfg.metric}={s:.4f} | interim={interim:.4f} | {dt:.1f}s")

            trial.report(interim, step=i)
            if trial.should_prune():
                raise optuna.TrialPruned(f"Pruned at fold {i} with interim {interim:.4f}")

        return float(np.nanmean(scores))

    def run(self):
        cfg = self.cfg

        sampler = optuna.samplers.TPESampler(seed=cfg.seed)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=5)

        study = optuna.create_study(
            direction="maximize",
            sampler=sampler,
            pruner=pruner,
            study_name=cfg.study_name,
        )

        study.optimize(self.objective, n_trials=cfg.n_trials, show_progress_bar=True)

        # Re-evaluate best params to get mean ± std explicitly
        bp = study.best_params

        best_dfo = DFOConfig(
            metric=bp["dfo_metric"],
            num_clusters=int(bp["dfo_num_clusters"]),
            order=bp["dfo_order"],
            mutation_prob=float(bp["dfo_mut_prob"]),
            tolerance=float(bp["dfo_tol"]),
            seed=cfg.seed,
        )
        best_embedding_dim = int(bp["embedding_dim"])
        best_backbone_kwargs = dict(
            hidden_size=int(bp["lstm_hidden_size"]),
            num_layers=int(bp["lstm_num_layers"]),
            dropout=float(bp["lstm_dropout"]),
        )

        best_mean, best_std, best_scores = self._eval_params(best_dfo, best_embedding_dim, best_backbone_kwargs)

        best = {
            "best_value_optuna_mean": float(study.best_value),
            "best_mean_5x5": best_mean,
            "best_std_5x5": best_std,
            "best_params": study.best_params,
            "n_trials": len(study.trials),
        }

        out_path = os.path.join(cfg.out_dir, f"{cfg.study_name}_best.json")
        with open(out_path, "w") as f:
            json.dump(best, f, indent=2)

        print("\n=== DONE ===")
        print(f"Optuna best (mean objective): {best['best_value_optuna_mean']:.4f}")
        print(f"Best 5x5 CV: {best_mean:.4f} ± {best_std:.4f}")
        print("Best params:", best["best_params"])
        print("Saved:", out_path)

        # Optional: show all 25 fold scores
        print("Best per-fold scores:", [round(x, 4) for x in best_scores])

        return study, best


# -----------------------------
# Run
# -----------------------------
cfg = OptunaColonLSTMConfig(
    csv_path="coloncancer_encoded.csv",
    label_col="label",
    repeats=5,
    n_splits=5,
    seed=42,
    n_trials=5,          
    epochs=120,
    batch_size=32,
    lr=1e-3,
    metric="acc",
    out_dir="./optuna_colon_lstm",
    study_name="colon_lstm_5x5_dynatab",
)

runner = OptunaColonLSTM5x5(cfg)
study, best = runner.run()

[I 2026-02-20 18:01:38,439] A new study created in memory with name: colon_lstm_5x5_dynatab


Device: cuda
Loaded: (62, 2000) | label counts: {0: 40, 1: 22}
Prepared 25 CV folds (5x5).


  0%|          | 0/5 [00:00<?, ?it/s]


[Trial 0] DFO(metric=euclidean, K=2, order=ascending, mut=0.61, tol=4.2e-03) | LSTM(emb=128, hidden=64, layers=1, drop=0.12)
  Fold 1/25 | acc=0.9231 | interim=0.9231 | 6.0s
  Fold 2/25 | acc=0.5385 | interim=0.7308 | 5.5s
  Fold 3/25 | acc=0.6667 | interim=0.7094 | 5.4s
  Fold 4/25 | acc=0.5833 | interim=0.6779 | 5.6s
  Fold 5/25 | acc=0.5833 | interim=0.6590 | 5.5s
  Fold 6/25 | acc=1.0000 | interim=0.7158 | 5.4s
  Fold 7/25 | acc=0.7692 | interim=0.7234 | 5.4s
  Fold 8/25 | acc=0.7500 | interim=0.7268 | 5.4s
  Fold 9/25 | acc=0.7500 | interim=0.7293 | 5.4s
  Fold 10/25 | acc=0.7500 | interim=0.7314 | 5.5s
  Fold 11/25 | acc=0.6154 | interim=0.7209 | 5.4s
  Fold 12/25 | acc=0.6923 | interim=0.7185 | 5.4s
  Fold 13/25 | acc=0.6667 | interim=0.7145 | 5.4s
  Fold 14/25 | acc=0.8333 | interim=0.7230 | 5.5s
  Fold 15/25 | acc=0.6667 | interim=0.7192 | 5.4s
  Fold 16/25 | acc=0.7692 | interim=0.7224 | 5.5s
  Fold 17/25 | acc=0.7692 | interim=0.7251 | 5.4s
  Fold 18/25 | acc=0.5833 | inter

C:\Users\ah00069\dynatab\dfo.py:167: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  metric_matrix = torch.var(X_tensor.unsqueeze(2) - X_tensor.unsqueeze(1), dim=0).abs()


  Fold 6/25 | acc=0.4615 | interim=0.6389 | 5.0s
  Fold 7/25 | acc=0.3846 | interim=0.6026 | 5.0s
  Fold 8/25 | acc=0.7500 | interim=0.6210 | 4.9s
  Fold 9/25 | acc=0.6667 | interim=0.6261 | 5.0s
  Fold 10/25 | acc=0.7500 | interim=0.6385 | 4.8s


C:\Users\ah00069\dynatab\dfo.py:167: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  metric_matrix = torch.var(X_tensor.unsqueeze(2) - X_tensor.unsqueeze(1), dim=0).abs()


  Fold 11/25 | acc=0.6923 | interim=0.6434 | 4.9s
  Fold 12/25 | acc=0.6154 | interim=0.6410 | 5.0s
  Fold 13/25 | acc=0.6667 | interim=0.6430 | 4.9s
  Fold 14/25 | acc=0.5833 | interim=0.6387 | 5.2s
  Fold 15/25 | acc=0.5833 | interim=0.6350 | 4.9s
  Fold 16/25 | acc=0.6923 | interim=0.6386 | 4.9s
  Fold 17/25 | acc=0.6923 | interim=0.6418 | 5.1s
  Fold 18/25 | acc=0.6667 | interim=0.6432 | 5.1s
  Fold 19/25 | acc=0.8333 | interim=0.6532 | 4.9s


C:\Users\ah00069\dynatab\dfo.py:167: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  metric_matrix = torch.var(X_tensor.unsqueeze(2) - X_tensor.unsqueeze(1), dim=0).abs()


  Fold 20/25 | acc=0.5833 | interim=0.6497 | 5.0s
  Fold 21/25 | acc=0.6154 | interim=0.6480 | 5.0s
  Fold 22/25 | acc=0.6923 | interim=0.6501 | 5.0s
  Fold 23/25 | acc=0.6667 | interim=0.6508 | 4.9s
  Fold 24/25 | acc=0.8333 | interim=0.6584 | 4.9s
  Fold 25/25 | acc=0.6667 | interim=0.6587 | 5.2s
[I 2026-02-20 18:08:17,797] Trial 2 finished with value: 0.6587179720401763 and parameters: {'dfo_metric': 'variance', 'dfo_num_clusters': 5, 'dfo_order': 'ascending', 'dfo_mut_prob': 0.34662383707788913, 'dfo_tol': 0.0001238264969702355, 'embedding_dim': 32, 'lstm_hidden_size': 256, 'lstm_num_layers': 1, 'lstm_dropout': 0.38783385110582347}. Best is trial 0 with value: 0.699487202167511.

[Trial 3] DFO(metric=euclidean, K=6, order=descending, mut=0.03, tol=7.6e-04) | LSTM(emb=128, hidden=256, layers=1, drop=0.32)
  Fold 1/25 | acc=0.6154 | interim=0.6154 | 5.4s
  Fold 2/25 | acc=0.5385 | interim=0.5769 | 5.4s
  Fold 3/25 | acc=0.5000 | interim=0.5513 | 5.4s
  Fold 4/25 | acc=0.8333 | interi